In [0]:
from pyspark.sql.functions import current_timestamp, expr

In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
checkpoint = '/Volumes/retail_store_dev/landing/retail_store/order_checkpoint/'
source_df = spark.readStream \
    .format("cloudFiles") \
        .option("cloudFiles.format", "json") \
            .option("cloudFiles.schemaLocation", checkpoint) \
                    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
            .load("/Volumes/retail_store_dev/landing/retail_store/orders/")    

In [0]:
source_df = source_df.withColumns({
    "load_date":current_timestamp(),
    "file_name":expr("_metadata.file_name")
    })
source_df.writeStream \
    .outputMode("append") \
    .format("delta") \
        .option("checkpointLocation", checkpoint) \
            .trigger(availableNow=True) \
            .option("mergeSchema", "true") \
                .toTable(f"{catalog}.bronze.orders")